In [13]:
import numpy as np
import json
from datetime import datetime
from typing import List, Dict, Tuple, Optional

In [14]:
"""
SBTI - Hybrid Calculation Method 
1. Không gian Vector 15 chiều
2. Reverse Scoring + Weighted Accumulation
3. Mahalanobis Distance (statistically superior)
4. Pattern Detection + Dimension Analysis
5. 27 Archetypes
"""

'\nSBTI - Hybrid Calculation Method \n1. Không gian Vector 15 chiều\n2. Reverse Scoring + Weighted Accumulation\n3. Mahalanobis Distance (statistically superior)\n4. Pattern Detection + Dimension Analysis\n5. 27 Archetypes\n'

In [ ]:
# ====================== DIMENSIONS ======================
DIMENSION_NAMES = [
    "S1_SelfEsteem", "S2_SelfClarity", "S3_Purpose",
    "E1_Attachment", "E2_EmotionalDepth", "E3_Independence",
    "A1_Worldview", "A2_RulesFlex", "A3_Meaning",
    "Ac1_Motivation", "Ac2_Decision", "Ac3_Execution",
    "So1_SocialProactivity", "So2_Boundaries", "So3_Authenticity"
]

DIMENSION_GROUPS = {
    "Bản Thân": [0, 1, 2],
    "Cảm Xúc": [3, 4, 5],
    "Thái Độ": [6, 7, 8],
    "Hành Động": [9, 10, 11],
    "Xã Hội": [12, 13, 14]
}

DIMENSION_DESCRIPTIONS = {
    "S1_SelfEsteem": {
        "L": "Luôn nghĩ bản thân kém cỏi, sợ sai, sợ bị đánh giá.",
        "M": "Biết bản thân có năng lực, nhưng đôi lúc vẫn tự hỏi 'liệu mình có tự tin quá không?', khiêm tốn đúng lúc.",
        "H": "Tin mình xuất sắc, hiếm khi nghi ngờ bản thân. Sai lầm là do hoàn cảnh, không phải do mình."
    },
    "S2_SelfClarity": {
        "L": "Không rõ mình thích gì, muốn gì, cảm xúc lộn xộn. Người ngoài đôi khi hiểu bạn hơn chính bạn.",
        "M": "Biết điểm mạnh, điểm yếu cơ bản, nhưng đôi khi vẫn bất ngờ với chính phản ứng của mình.",
        "H": "Biết rõ mình là ai, muốn gì, cảm xúc ra sao và vì sao."
    },
    "S3_Purpose": {
        "L": "Không rõ điều gì thực sự quan trọng với mình. Dễ bị ảnh hưởng bởi người khác, xã hội.",
        "M": "Có ý thức về những gì mình coi trọng. Đôi khi phân vân giữa điều mình thực sự muốn và điều người khác kỳ vọng.",
        "H": "Biết rõ điều gì là quan trọng nhất với mình và chủ động sống theo điều đó. Quyết định nhất quán với giá trị cốt lõi."
    },
    "E1_Attachment": {
        "L": "Nhạy cảm, bất an trong gắn bó. Luôn nghĩ sẽ bị đâm sau lưng bất cứ lúc nào.",
        "M": "Nửa tin nửa ngờ. Trong trạng thái mơ hồ trong các mối quan hệ.",
        "H": "Tin vào quan hệ của chính mình. Khi gặp được định mệnh đời mình thì 100% tin vào đối phương."
    },
    "E2_EmotionalDepth": {
        "L": "Kiềm chế, giữ khoảng cách. Luôn có bức tường ngăn cản sự trao đi về mặt cảm xúc.",
        "M": "Biết cân bằng giữa tình và lý. Luôn luôn có song luồng ý kiến để cân bằng cảm xúc.",
        "H": "Dồn nhiều cảm xúc. Chia sẻ quá nhiều chi tiết cá nhân/cảm xúc sâu kín ngay khi mới quen."
    },
    "E3_Independence": {
        "L": "Cởi mở, chào đón với người khác. Dễ bắt chuyện, biết lắng nghe.",
        "M": "Cân bằng giữa gần gũi và độc lập. Thân thiện nhưng vẫn cần không gian riêng.",
        "H": "Độc lập. Khó mở lòng, cần nhiều không gian riêng, không thích bị kiểm soát."
    },
    "A1_Worldview": {
        "L": "Tập trung vào việc phòng ngừa rủi ro, duy trì sự an toàn.",
        "M": "Ưu tiên quan sát và thu thập đầy đủ dữ liệu trước khi đưa ra phản ứng.",
        "H": "Tập trung vào việc tìm kiếm phương án mới, tin vào khả năng cải thiện kết quả."
    },
    "A2_RulesFlex": {
        "L": "Ưu tiên các phương pháp truyền thống và những giải pháp đã được kiểm chứng.",
        "M": "Điều chỉnh cách làm tùy theo hoàn cảnh và thông tin thực tế nhận được.",
        "H": "Ưu tiên thử nghiệm các cách tiếp cận mới, không ngại thay đổi hoàn toàn quy trình."
    },
    "A3_Meaning": {
        "L": "Thường phản ứng theo tình huống thực tế, ưu tiên sự tự nhiên và linh hoạt.",
        "M": "Có sự kết hợp giữa việc duy trì mục tiêu cá nhân và khả năng điều chỉnh định hướng.",
        "H": "Có sự gắn kết chặt chẽ với mục đích cá nhân, luôn xác định rõ lý do của các hành động."
    },
    "Ac1_Motivation": {
        "L": "Hành động chỉ để né rủi ro, giữ ổn định. Luôn nghĩ đến cái xấu nhất trước.",
        "M": "Vừa muốn an toàn vừa muốn thành tựu, nhưng không quá liều.",
        "H": "Hành động vì thành tựu, vinh quang, trải nghiệm. Sẵn sàng đánh đổi, không sợ thất bại."
    },
    "Ac2_Decision": {
        "L": "Rất khó đưa ra quyết định, sợ chọn sai. Cần hỏi ý kiến mọi người.",
        "M": "Quyết định khi đã có đủ thông tin cơ bản, nhưng vẫn cần chút thời gian cân nhắc.",
        "H": "Quyết đoán, ít lăn tăn. Tin vào phán đoán của mình, sai thì sửa."
    },
    "Ac3_Execution": {
        "L": "Trì hoãn kinh niên, luôn để công việc đến phút cuối. Vô số lý do để 'để mai tính'.",
        "M": "Cần một chút deadline hoặc áp lực bên ngoài để bắt đầu, nhưng vẫn hoàn thành đúng hạn.",
        "H": "Làm xong trước hạn, tự giác, không cần ai nhắc. Ghét cảm giác bị deadline dí."
    },
    "So1_SocialProactivity": {
        "L": "Thường dè dặt, không nhanh nhạy tham gia các hoạt động xã hội. Cần thời gian để chuẩn bị.",
        "M": "Cẩn trọng, tham gia khi hiểu rõ tình huống và phản ứng hợp lý.",
        "H": "Chủ động, tích cực tương tác với người khác, sẵn sàng thử các cách tiếp cận mới."
    },
    "So2_Boundaries": {
        "L": "Dễ bị ảnh hưởng, khó từ chối, thường chiều theo người khác.",
        "M": "Thân thiện nhưng có chọn lọc, biết khi nào gần, khi nào giữ riêng tư.",
        "H": "Cảnh giác cao, giữ ranh giới rõ ràng, bảo vệ bản thân."
    },
    "So3_Authenticity": {
        "L": "Điều chỉnh hành vi để hòa nhập, chịu tác động từ người khác.",
        "M": "Cân bằng giữa thể hiện bản thân và xã hội, cảm xúc vừa phải.",
        "H": "Chân thật, mạnh mẽ, tự tin, vẫn tôn trọng người khác và trật tự xã hội."
    }
}

# ====================== DIMENSION WEIGHTS ======================
DIMENSION_WEIGHTS = np.array([
    1.25, 1.30, 1.35, 1.20, 1.40, 1.15, 1.10, 1.05, 
    1.20, 1.50, 1.55, 1.60, 1.10, 1.25, 1.15
])
DIMENSION_WEIGHTS = DIMENSION_WEIGHTS / np.sum(DIMENSION_WEIGHTS)

# ====================== 27 ARCHETYPES (Updated with correct vectors) ======================
# Pattern format: "HHH-HMH-MHH..." converts to [2,2,2,1,2,1,2,2,1,...]
# where: L=0, M=1, H=2
ARCHETYPES = {
    "CTRL":  {"vector": [2,2,2,1,2,1,2,2,1,2,2,2,2,1,2], "desc": "The Controller - Trùm Kiếm Soát", "code": "CTRL"},
    "ATMR":  {"vector": [2,2,2,1,1,2,2,2,1,1,2,1,2,1,0], "desc": "ATM-er - Cây ATM Bẻ Di", "code": "ATMR"},
    "DIOR":  {"vector": [1,2,1,2,2,1,2,1,2,1,2,1,0,1,0], "desc": "Dior-s - Kẻ Thất Bại", "code": "DIOR"},
    "BOSS":  {"vector": [2,2,2,1,2,1,2,2,1,2,2,2,2,1,0], "desc": "The Boss - Thủ Lĩnh", "code": "BOSS"},
    "THANK": {"vector": [1,2,1,1,2,2,2,1,2,2,2,1,2,1,0], "desc": "THAN-K - Người Biết Ơn", "code": "THANK"},
    "OHNO":  {"vector": [2,2,0,0,2,1,0,2,1,2,2,1,2,1,0], "desc": "OH-NO - Người OH-NO", "code": "OHNO"},
    "GOGO":  {"vector": [2,2,1,1,2,1,2,2,1,2,2,2,2,1,2], "desc": "GOGO - Người Go-Go", "code": "GOGO"},
    "SEXY":  {"vector": [1,2,1,1,1,0,1,2,2,1,2,2,1,0,1], "desc": "SEXY - Người Hấp Dẫn", "code": "SEXY"},
    "LOVR":  {"vector": [1,0,1,0,1,0,2,0,1,2,0,2,2,0,1], "desc": "LOVE-R - Người Lãng Mạn", "code": "LOVR"},
    "MUMM":  {"vector": [1,2,1,2,1,0,2,1,2,0,2,2,2,0,0], "desc": "MUM - Mẹ", "code": "MUMM"},
    "FAKE":  {"vector": [2,0,1,2,2,0,1,0,2,2,0,2,2,0,1], "desc": "FAKE - Người Giả", "code": "FAKE"},
    "OJBK":  {"vector": [1,2,1,2,2,2,2,1,0,0,2,2,2,2,0], "desc": "OJBK - Người Tùy Tiện", "code": "OJBK"},
    "MALO":  {"vector": [1,0,1,2,1,2,2,0,1,2,0,1,0,2,1], "desc": "MALO - Khi Nho", "code": "MALO"},
    "JOKER": {"vector": [0,0,1,0,1,0,0,2,0,0,0,0,2,0,2], "desc": "JOKE-R - Người Hề", "code": "JOKER"},
    "WOCI":  {"vector": [2,2,0,1,2,1,2,2,1,2,1,2,0,1,1], "desc": "WOC! - Người WOC!", "code": "WOCI"},
    "THINK": {"vector": [2,2,0,1,2,1,2,0,1,2,1,2,0,1,1], "desc": "THIN-K - Người Suy Tư", "code": "THINK"},
    "SHIT":  {"vector": [2,2,0,1,0,1,0,2,2,2,1,2,0,1,1], "desc": "SHIT - Người Hận Thù", "code": "SHIT"},
    "ZZZZ":  {"vector": [1,1,0,2,0,1,0,2,0,2,2,0,0,1,2], "desc": "ZZZZ - Người Ma", "code": "ZZZZ"},
    "POOR":  {"vector": [2,2,0,2,0,1,0,2,1,2,2,2,0,1,0], "desc": "POOR - Người Nghèo", "code": "POOR"},
    "MONK":  {"vector": [2,2,0,0,0,1,0,0,2,2,2,0,0,1,2], "desc": "MONK - Nhà Sư", "code": "MONK"},
    "IMSB":  {"vector": [0,0,1,0,2,2,0,0,0,0,0,0,2,0,2], "desc": "IMSB - Người Ngu", "code": "IMSB"},
    "SOLO":  {"vector": [0,2,0,0,0,2,0,1,0,0,2,0,0,2,2], "desc": "SOLO - Người Cô Đơn", "code": "SOLO"},
    "FUCK":  {"vector": [1,0,0,0,1,0,0,0,2,2,0,0,2,0,1], "desc": "FUCK - Người Hoang Dã", "code": "FUCK"},
    "DEAD":  {"vector": [0,0,0,0,0,2,0,2,0,0,0,0,0,1,2], "desc": "DEAD - Kẻ Chết", "code": "DEAD"},
    "IMFW":  {"vector": [0,0,1,0,1,0,0,1,0,0,0,0,2,0,0], "desc": "IMFW - Người Vô Dụng", "code": "IMFW"},
    "DRUNK": {"vector": [2,2,2,2,2,2,2,2,2,2,2,2,2,2,2], "desc": "DRUNK - Người Say Xỉn (Hidden)", "code": "DRUNK"},
    "HHHH":  {"vector": [1,1,1,1,1,1,1,1,1,1,1,1,1,1,1], "desc": "HHHH - Người Giả Chết (Fallback)", "code": "HHHH"},
}

WEIGHTS = DIMENSION_WEIGHTS  # alias for compatibility

In [ ]:
# ====================== MATCHING SETUP ======================
# Logic:
# 1. User answers 30 questions (2 per dimension)
# 2. Average scores per dimension (0-2 range)
# 3. Calculate distance to each archetype
# 4. Convert distance to similarity %
# 5. Rank archetypes by similarity
# 6. Calculate confidence based on score gap

✓ Simplified matching engine initialized (Euclidean distance)


In [ ]:
class SBTI_Hybrid:
    """Complete SBTI Implementation with 27 Archetypes and 15 Dimensions"""
    
    def __init__(self):
        self.archetype_names = list(ARCHETYPES.keys())
        self.archetype_vectors = np.array([ARCHETYPES[name]["vector"] for name in self.archetype_names])
        self.weights = DIMENSION_WEIGHTS
        
    def get_questions(self) -> List[Dict]:
        """Return 30 questions from the official SBTI table"""
        questions = [
            # BẢN THÂN (Self)
            {"num": 1, "dim_idx": 0, "dim": "S1_SelfEsteem", "q": "Đối tượng hẹn hò của bạn là người hiếu thảo với cha mẹ, thương yêu trẻ nhỏ, hiền lành thật thà, sống trong sạch, tính tình ngay thẳng, tài giỏi hết chỗ chê, nói chuyện có duyên, tinh tế trong quan sát, đọc nhiều biết rộng, kiên nhẫn chỉ bảo, dễ gần dễ mến, tốt bụng hết sức, đầy tham vọng, phong độ ngời ngời, mặt mũi đẹp trai/xinh gái cỡ idol — nói chung là con người hoàn hảo vi phạm quy luật tự nhiên. Lúc này bạn sẽ?", "opts": ["Dù bạn ấy có tuyệt vời thế nào, tôi cũng không để mất mình.", "Ở giữa A và C", "Tôi sẽ trân trọng bạn ấy hết mình. Có thể sẽ biến thành simp"], "reverse": False},
            {"num": 2, "dim_idx": 0, "dim": "S1_SelfEsteem", "q": "Tôi không đủ giỏi. Mọi người xung quanh đều giỏi hơn tôi.", "opts": ["Đúng vậy", "Thi thoảng", "Không hẳn"], "reverse": True},
            
            {"num": 3, "dim_idx": 1, "dim": "S2_SelfClarity", "q": "Đánh giá của người ngoài? Tôi kệ mẹ nó.", "opts": ["Không đồng ý", "Trung lập", "Đồng ý"], "reverse": False},
            {"num": 4, "dim_idx": 1, "dim": "S2_SelfClarity", "q": "Tôi không chỉ là kẻ thất bại, tôi còn là thằng hề, là con cá lên thớt. Cả đời chưa từng yêu ai. Nhát gan và tự ti. Tuổi thanh xuân của tôi chỉ là một chuỗi phim viễn tưởng — tưởng tượng sẽ có ai đó muốn cùng tôi đi dạo phố, cùng đi chơi, làm mấy thứ bình thường mà cặp đôi nào cũng làm. Thực tế? Đốt sạch tiền bố mẹ, học trường rác, rồi trôi dạt vào một công việc bế tắc. Không ước mơ, không mục tiêu, không kỹ năng — con số không ba lần. Mỗi lần thấy người ta đùa về mấy thằng loser trên mạng, tôi muốn khóc. Tôi là con chuột cống nhìn lên qua khe nắp cống, ngắm cuộc sống tươi đẹp của người ta. Mỗi lần nhìn thấy là thêm một nhát dao vào tim, thêm một mét vuông không gian sống bị nén lại. Xin hãy cho bọn hề như chúng tôi một con đường sống. Tôi thật sự không muốn khóc ướt gối giữa ban ngày nữa.", "opts": ["Tôi khóc rồi...", "Cái gì thế này...", "Đây không phải tôi!"], "reverse": True},
            
            {"num": 5, "dim_idx": 2, "dim": "S3_Purpose", "q": "Tôi hiểu rõ bản thân mình thật sự là người như thế nào.", "opts": ["Không đồng ý", "Trung lập", "Đồng ý"], "reverse": False},
            {"num": 6, "dim_idx": 2, "dim": "S3_Purpose", "q": "Trong sâu thẳm, tôi có một thứ gì đó mà tôi thật sự muốn theo đuổi.", "opts": ["Không đồng ý", "Trung lập", "Đồng ý"], "reverse": False},
            
            # CẢM XÚC (Emotion)
            {"num": 7, "dim_idx": 3, "dim": "E1_Attachment", "q": "Trong tình cảm, tôi thường xuyên lo sợ bị bỏ rơi.", "opts": ["Đúng", "Thi thoảng", "Không"], "reverse": True},
            {"num": 8, "dim_idx": 3, "dim": "E1_Attachment", "q": "Người yêu bạn hơn 5 tiếng không trả lời tin nhắn, bảo là bị ngộ độc thực phẩm. Bạn nghĩ sao?", "opts": ["Ngộ độc gì mà tới 5 tiếng. Chắc đang giấu gì rồi.", "Dao động giữa tin và nghi.", "Có lẽ hôm nay bạn ấy thật sự khó chịu."], "reverse": False},
            
            {"num": 9, "dim_idx": 4, "dim": "E2_EmotionalDepth", "q": "Tôi thề có trời đất, tôi đối xử với mọi mối quan hệ tình cảm đều nghiêm túc!", "opts": ["Thật ra là không", "Có lẽ?", "Đúng! (lương tâm trong sáng, ngẩng cao đầu)"], "reverse": False},
            {"num": 10, "dim_idx": 4, "dim": "E2_EmotionalDepth", "q": "Tôi khao khát được thân thiết với những người mình tin tưởng — như người thân thất lạc lâu năm.", "opts": ["Không đồng ý", "Trung lập", "Đồng ý"], "reverse": False},
            
            {"num": 11, "dim_idx": 5, "dim": "E3_Independence", "q": "Tôi coi trọng không gian cá nhân trong mọi mối quan hệ.", "opts": ["Tôi thích được phụ thuộc lẫn nhau hơn", "Tùy tình huống", "Đúng! (nói một cách dứt khoát)"], "reverse": False},
            {"num": 12, "dim_idx": 5, "dim": "E3_Independence", "q": "Sau khi yêu nhau, đối phương cực kỳ dính người. Bạn cảm thấy thế nào?", "opts": ["Sướng quá đi chứ", "Sao cũng được", "Tôi thích giữ không gian riêng hơn"], "reverse": False},
            
            # THÁI ĐỘ (Attitude)
            {"num": 13, "dim_idx": 6, "dim": "A1_Worldview", "q": "Bạn đang đi trên phố. Một cô bé siêu dễ thương nhảy chân sáo đi về phía bạn (dễ thương từ mọi góc nhìn, chụp bằng điện thoại nào cũng dễ thương, dễ thương đến chết đi được). Cô bé đưa cho bạn một cây kẹo mút. Bạn phản ứng thế nào?", "opts": ["Có khi là lừa đảo kiểu mới? Tốt nhất nên đi cho lành.", "Ngơ ngác hoàn toàn, gãi đầu", "Ôi cô bé dễ thương quá! Cô bé cho tôi kẹo mút nè!"], "reverse": False},
            {"num": 14, "dim_idx": 6, "dim": "A1_Worldview", "q": "Đa số mọi người đều thiện lương.", "opts": ["Thật ra trái tim độc ác trên đời này còn nhiều hơn cả bệnh trĩ.", "Có lẽ", "Đúng, tôi chọn tin rằng người tốt nhiều hơn."], "reverse": False},
            
            {"num": 15, "dim_idx": 7, "dim": "A2_RulesFlex", "q": "Tôi thích phá vỡ khuôn khổ. Tôi ghét bị ràng buộc.", "opts": ["Đồng ý", "Trung lập", "Không đồng ý"], "reverse": False},
            {"num": 16, "dim_idx": 7, "dim": "A2_RulesFlex", "q": "Tôi thường lên kế hoạch, ____", "opts": ["Nhưng kế hoạch luôn không theo kịp thay đổi", "Có khi hoàn thành, có khi không.", "Tôi ghét bị phá vỡ kế hoạch."], "reverse": False},
            
            {"num": 17, "dim_idx": 8, "dim": "A3_Meaning", "q": "Tôi làm việc thường có mục tiêu.", "opts": ["Không đồng ý", "Trung lập", "Đồng ý"], "reverse": False},
            {"num": 18, "dim_idx": 8, "dim": "A3_Meaning", "q": "Bất chợt một ngày, tôi nhận ra cuộc đời chẳng có ý nghĩa chó gì. Con người chẳng qua cũng chỉ là động vật bị ham muốn điều khiển — hoàn toàn là cỗ máy thịt chạy bằng hormone. Đói thì ăn. Buồn ngủ thì ngủ. Lên cơn thì... bạn hiểu mà. Chúng ta chẳng khác gì gia súc.", "opts": ["Đúng là vậy.", "Có lẽ đúng, có lẽ không.", "Nói bậy nói bạ."], "reverse": True},
            
            # HÀNH ĐỘNG (Action)
            {"num": 19, "dim_idx": 9, "dim": "Ac1_Motivation", "q": "Tôi nhất định phải không ngừng leo lên, trở nên mạnh mẽ hơn.", "opts": ["Không đồng ý", "Trung lập", "Đồng ý"], "reverse": False},
            {"num": 20, "dim_idx": 9, "dim": "Ac1_Motivation", "q": "Tôi làm việc chủ yếu để đạt kết quả và tiến bộ, không phải để tránh rắc rối và rủi ro.", "opts": ["Không đồng ý", "Trung lập", "Đồng ý"], "reverse": False},
            
            {"num": 21, "dim_idx": 10, "dim": "Ac2_Decision", "q": "Tôi quyết định khá dứt khoát, không thích do dự.", "opts": ["Không đồng ý", "Trung lập", "Đồng ý"], "reverse": False},
            {"num": 22, "dim_idx": 10, "dim": "Ac2_Decision", "q": "Bạn đã ngồi trên bồn cầu 30 phút rồi, táo bón khổ sở. Chẳng có gì xảy ra cả. Bạn sẽ làm gì?", "opts": ["Ngồi thêm 30 phút nữa. Biết đâu sẽ ra", "Vỗ mông mình và hét: 'Mày ra đi, đồ khốn!'", "Dùng thuốc nhuận tràng. Cho xong nhanh đi."], "reverse": False},
            
            {"num": 23, "dim_idx": 11, "dim": "Ac3_Execution", "q": "Sắp thi rồi. Trường bắt buộc phải đi học tối — nghỉ là bị trừ điểm. Nhưng tối nay bạn đã hẹn crush cùng chơi Liên Quân. Bạn sẽ làm gì?", "opts": ["Trốn! Có một tối thôi mà!", "Thôi xin nghỉ phép luôn", "Sắp thi rồi còn đi đâu."], "reverse": False},
            {"num": 24, "dim_idx": 11, "dim": "Ac3_Execution", "q": "Ai đó khen bạn 'khả năng thực thi cao'. Câu nào gần nhất với cảm giác của bạn?", "opts": ["Khả năng thực thi của tôi chỉ cao khi bị deadline dí...", "Ờ, thỉnh thoảng.", "Đúng - việc thì phải đẩy đi chứ."], "reverse": False},
            
            # XÃ HỘI (Social)
            {"num": 25, "dim_idx": 12, "dim": "So1_SocialProactivity", "q": "Bạn của bạn dẫn theo bạn của họ đi chơi cùng. Trạng thái của bạn thường là gì?", "opts": ["Tôi tự nhiên giữ khoảng cách với 'bạn của bạn' — sợ làm hỏng mối quan", "Tùy người. Hơp thì chơi.", "Bạn của bạn cũng là bạn tôi! Thời gian giao lưu đây!"], "reverse": False},
            {"num": 26, "dim_idx": 12, "dim": "So1_SocialProactivity", "q": "Bạn kết bạn online qua game và được mời gặp ngoài đời. Bạn nghĩ sao?", "opts": ["Nói chuyện trên mạng thì OK, nhưng gặp ngoài đời thì hơi run.", "Cũng được. Ai muốn chat thì chat.", "Tôi sẽ ăn mặc đẹp và tự tin giao lưu. Biết đâu... tôi nói biết đâu thôi nhé?"], "reverse": False},
            
            {"num": 27, "dim_idx": 13, "dim": "So2_Boundaries", "q": "Cách tôi xử lý các mối quan hệ giống hệt hàng rào điện tử vô hình. Lại gần quá là báo động.", "opts": ["Không đồng ý", "Trung lập", "Đồng ý"], "reverse": False},
            {"num": 28, "dim_idx": 13, "dim": "So2_Boundaries", "q": "Đôi khi bạn có ý kiến tiêu cực về một việc gì đó nhưng không nói ra. Đa phần lý do là gì?", "opts": ["Tình huống đó hiếm khi xảy ra với tôi.", "Có lẽ vì nể mặt hoặc giữ quan hệ.", "Tôi không muốn người khác biết mình có mặt tối."], "reverse": False},
            
            {"num": 29, "dim_idx": 14, "dim": "So3_Authenticity", "q": "Tôi cư xử khác nhau tùy thuộc vào đối tượng.", "opts": ["Đồng ý", "Trung lập", "Không đồng ý"], "reverse": False},
            
            # Bonus - Sở thích
            {"num": 30, "dim_idx": -1, "dim": "BONUS", "q": "Sở thích của bạn là gì?", "opts": ["Ăn ngủ thở", "Nghệ thuật", "Nhậu"], "reverse": False}
        ]
        
        return questions
    
    def take_test(self) -> Tuple[np.ndarray, bool]:
        """Interactive test administration"""
        questions = self.get_questions()
        answers = np.zeros(15)
        answer_counts = np.zeros(15)
        bonus_drink = False
        
        print("\n" + "="*80)
        print("SBTI - Silly Big Personality Test")
        print("="*80)
        print("Hướng dẫn: Trả lời 30 câu hỏi")
        print("(0 = Tùy chọn A, 1 = Tùy chọn B, 2 = Tùy chọn C)\n")
        
        for q in questions:
            print(f"\nCâu {q['num']}: {q['q']}")
            for i, opt in enumerate(q['opts']):
                print(f"  {i}. {opt}")
            
            while True:
                try:
                    ans = input("Nhập lựa chọn (0-2): ").strip()
                    ans = int(ans)
                    if ans not in [0, 1, 2]:
                        print("Vui lòng nhập 0, 1 hoặc 2")
                        continue
                    break
                except ValueError:
                    print("Vui lòng nhập số")
            
            # Bonus question
            if q['dim'] == "BONUS":
                bonus_drink = ans == 2  # True if "Nhậu"
            else:
                # Apply reverse scoring if needed
                if q['reverse']:
                    ans = 2 - ans
                
                answers[q['dim_idx']] += ans
                answer_counts[q['dim_idx']] += 1
        
        # Average scores for each dimension
        user_vector = answers / answer_counts
        return user_vector, bonus_drink
    
    def distance_match(self, user_vector: np.ndarray) -> Tuple[List[Dict], str]:
        """
        Calculate Euclidean distance to all archetypes
        Convert to similarity percentage using exponential decay
        """
        user_weighted = user_vector * self.weights
        
        distances = []
        for i, arch_name in enumerate(self.archetype_names):
            arch_vector = self.archetype_vectors[i]
            arch_weighted = arch_vector * self.weights
            
            # Euclidean distance
            dist = np.sqrt(np.sum((user_weighted - arch_weighted) ** 2))
            distances.append(dist)
        
        distances = np.array(distances)
        
        # Check for DRUNK trigger (all answers were 2)
        if np.all(user_vector >= 1.95):
            top_idx = self.archetype_names.index("DRUNK")
        elif np.all(np.abs(user_vector - 1.0) < 0.01):  # All near 1
            top_idx = self.archetype_names.index("HHHH")
        else:
            top_idx = np.argmin(distances)
        
        # Sort by distance
        sorted_indices = np.argsort(distances)
        
        # Convert distance to similarity % with exponential decay
        factor = 4.5
        similarities = 100 * np.exp(-factor * (distances / distances.max()))
        
        results = []
        for idx in sorted_indices:
            results.append({
                "type": self.archetype_names[idx],
                "code": ARCHETYPES[self.archetype_names[idx]]["code"],
                "desc": ARCHETYPES[self.archetype_names[idx]]["desc"],
                "score": similarities[idx],
                "distance": distances[idx]
            })
        
        # Calculate confidence
        if len(results) >= 2:
            gap = results[0]["score"] - results[1]["score"]
            if gap >= 10:
                confidence = "HIGH"
            elif gap >= 5:
                confidence = "MEDIUM"
            else:
                confidence = "LOW"
        else:
            confidence = "MEDIUM"
        
        return results, confidence
    
    def analyze_dimensions(self, scores: np.ndarray) -> Dict:
        analysis = {}
        for group_name, indices in DIMENSION_GROUPS.items():
            group_scores = scores[list(indices)]
            avg = np.mean(group_scores)
            
            # Level classification
            if avg >= 1.33:
                level = "HIGH"
                level_short = "H"
            elif avg >= 0.67:
                level = "MEDIUM"
                level_short = "M"
            else:
                level = "LOW"
                level_short = "L"
            
            # Get dimension descriptions
            dim_descriptions = []
            for idx in indices:
                dim_name = DIMENSION_NAMES[idx]
                if dim_name in DIMENSION_DESCRIPTIONS:
                    desc = DIMENSION_DESCRIPTIONS[dim_name][level_short]
                    dim_descriptions.append(desc)
            
            analysis[group_name] = {
                "level": level,
                "score": round(avg * 50, 1),  # Scale to 0-100
                "raw": [round(x, 2) for x in group_scores],
                "descriptions": dim_descriptions
            }
        return analysis

In [ ]:
# ==================== MAIN EXECUTION ====================
sbti = SBTI_Hybrid()
user_vector, bonus_drink = sbti.take_test()

print("\n" + "="*90)
print("ANALYZING YOUR RESULTS (Improved Euclidean Matching)...")
print("="*90)

results, confidence = sbti.distance_match(user_vector)

# Get final result
final_type = results[0]["type"]
final_score = results[0]["score"]

# ==================== DISPLAY RESULTS ====================
print("\n" + "="*90)
print("YOUR MAIN TYPE")
print("="*90)
print(f"\n{final_type} ({final_score:.1f}%)")
print(ARCHETYPES[final_type]["desc"])
print(f"Confidence: {confidence}")
print(f"Method: Simple Euclidean Distance\n")

print("DIMENSION BREAKDOWN")
print("="*90)
dim_analysis = sbti.analyze_dimensions(user_vector)
for group, data in dim_analysis.items():
    raw_str = " ".join([['L','M','H'][min(2, int(round(x/0.67)))] for x in data['raw']])
    print(f"\n{group:12} → {data['level']:8} ({data['score']:5.1f}%)")
    print(f"   DNA: {raw_str}")
    for i, desc in enumerate(data['descriptions']):
        dim_idx = DIMENSION_GROUPS[group][i]
        dim_short = DIMENSION_NAMES[dim_idx].split('_')[0]
        print(f"   • {dim_short}: {desc}")

print("\n" + "="*90)
print("TOP 5 MATCHES")
print("="*90)
for i, r in enumerate(results[:5], 1):
    marker = "[1]" if i == 1 else f"[{i}]"
    print(f"{marker} {r['code']:6} {r['score']:6.1f}% - {r['desc']}")

print("\n" + "="*90)
print("YOUR DNA (15 Dimensions):")
print("="*90)
print("   " + " ".join(['L','M','H'][int(round(x/0.67))] for x in user_vector))

# Save result
result_data = {
    "timestamp": datetime.now().isoformat(),
    "main_type": final_type,
    "score": final_score,
    "confidence": confidence,
    "method": "Euclidean Distance (Improved)",
    "user_vector": user_vector.tolist(),
    "bonus_drink": bonus_drink,
    "top_matches": results[:5],
    "dimension_analysis": {k: {"level": v["level"], "score": v["score"]} for k, v in dim_analysis.items()}
}

with open("sbt_result.json", "w", encoding="utf-8") as f:
    json.dump(result_data, f, ensure_ascii=False, indent=2)

print("\n\nResults saved to sbt_result.json")
print("Test completed!")

=== SBTI - Hybrid Method v2.4 (Improved Scoring) ===


QUESTION  1 / 31
Trong chuyện tình cảm, tôi thường cảm thấy mình không đủ tốt so với những người yêu cũ của người ấy.

   0 - Đúng vậy, tôi hay bị ám ảnh bởi điều đó.
   1 - Thỉnh thoảng tôi mới nghĩ vậy khi mọi thứ không suôn sẻ.
   2 - Không, tôi tin vào giá trị của mình ở hiện tại.
------------------------------------------------------------------------------------------
   OK - Saved for S1_SelfEsteem

QUESTION  2 / 31
Người yêu bạn rủ bạn về ra mắt gia đình, nhưng bạn biết gia đình họ có điều kiện và rất khó tính.

   0 - Sợ mình không đủ tốt, kiểu gì cũng bị chê.
   1 - Tôi lo một chút, chuẩn bị trước vài thứ để tự tin hơn.
   2 - Tôi tự tin mình sẽ ghi điểm.
------------------------------------------------------------------------------------------
   OK - Saved for S1_SelfEsteem

QUESTION  3 / 31
Tôi là ai? Tôi thích gì? Tôi giỏi gì? Tôi muốn gì? Đừng hỏi tôi...

   0 - Ủa, sao giống mình thế...
   1 - Mình đang đọc cái * th